In [ ]:
import json
import math
import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from accelerate import Accelerator
from accelerate.utils import set_seed
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, get_peft_model
from score import score_wer
from torch.utils.data import DataLoader, SequentialSampler
from tqdm.auto import tqdm
from transformers import AutoModelForCTC, AutoProcessor, get_linear_schedule_with_warmup

In [ ]:
# Core config (in-notebook)
BASE_MODEL_ID = "nvidia/parakeet-ctc-1.1b"
PROCESSOR_FROM_PRETRAINED_KWARGS = {}
DATASET_ID = "quinnlue/asr-test"
AUDIO_COLUMN = "audio_path"
TEXT_COLUMN = "orthographic_text"
OUTPUT_DIR = Path("artifacts/parakeet-ctc-lora")
SAMPLE_RATE = 16000

# Training hyperparameters
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.005
WARMUP_STEPS = 100
NUM_TRAIN_EPOCHS = 3
MAX_GRAD_NORM = 1.0
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.98
ADAM_EPSILON = 1e-8
GRADIENT_CHECKPOINTING = True
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 2
SEED = 42

# LoRA params
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LORA_BIAS = "none"
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Optional smoke test
SMOKE_TEST = False
SMOKE_TRAIN_EXAMPLES = 64
SMOKE_EVAL_EXAMPLES = 16

# Runtime
DTYPE = torch.bfloat16

# Weights & Biases
WANDB_CFG = {
    "project": "parakeet-ctc-lora",
    "log_model": "false",
    "api_key_env_var": "WANDB_API_KEY",
}
WANDB_API_KEY_ENV_VAR = WANDB_CFG["api_key_env_var"]
if WANDB_API_KEY_ENV_VAR and os.getenv(WANDB_API_KEY_ENV_VAR):
    os.environ["WANDB_API_KEY"] = os.getenv(WANDB_API_KEY_ENV_VAR)

# Augmentation config (kept in notebook for future use)
AUGMENTATION_CFG = {
    "waveform": {
        "time_stretch_prob": 0.3,
        "rir_prob": 0.5,
        "noise_prob": 0.5,
        "min_speed": 0.9,
        "max_speed": 1.1,
        "max_stretch_samples": 480000,
        "min_snr_db": 5.0,
        "max_snr_db": 30.0,
    },
    "vtlp": {
        "prob": 0.3,
        "alpha_min": 0.9,
        "alpha_max": 1.1,
        "fhi": 4800.0,
    },
    "spec": {
        "prob": 1.0,
        "policy": "LB",
    },
    "noise_dataset": None,
    "rir_dataset": None,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if WANDB_CFG.get("project") is not None:
    os.environ["WANDB_PROJECT"] = str(WANDB_CFG["project"])
if WANDB_CFG.get("log_model") is not None:
    os.environ["WANDB_LOG_MODEL"] = str(WANDB_CFG["log_model"])

In [ ]:
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, **PROCESSOR_FROM_PRETRAINED_KWARGS)
model = AutoModelForCTC.from_pretrained(
    BASE_MODEL_ID,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
)

ds = load_dataset(DATASET_ID)
train_dataset = ds["train"].filter(lambda x: x['audio_duration_sec'] < 15)
eval_dataset = ds["validation"]
test_dataset = ds["test"]

if SMOKE_TEST:
    train_dataset = train_dataset.select(range(SMOKE_TRAIN_EXAMPLES))
    eval_dataset = eval_dataset.select(range(SMOKE_EVAL_EXAMPLES))

for name, split in [("train", train_dataset), ("eval", eval_dataset), ("test", test_dataset)]:
    print(name, len(split))

In [ ]:
@dataclass
class CTCDataCollator:
    processor: AutoProcessor
    audio_column: str
    text_column: str
    sampling_rate: int
    dtype: torch.dtype
    pad_token_id: int = 1024  # model's pad_token_id Ã¢â‚¬â€ used to mask label padding
    subsampling_factor: int = 8  # FastConformer conv subsampling ratio

    def __call__(self, batch):
        audio = [item[self.audio_column]["array"] for item in batch]
        text = [item[self.text_column] for item in batch]

        inputs = self.processor(
            audio=audio,
            sampling_rate=self.sampling_rate,
            return_tensors="pt",
            padding=True,
        )

        labels = self.processor.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
        )

        label_ids = labels["input_ids"]
        # Ã¢Å¡Â  Must use the model's pad_token_id (1024), NOT -100.
        # The model computes target_lengths as (labels != pad_token_id).sum(),
        # so using -100 causes it to count padding tokens as real labels,
        # violating the CTC length constraint and causing CUDA illegal memory access.
        label_ids = label_ids.masked_fill(labels["attention_mask"].ne(1), self.pad_token_id)

        # ---- CTC length guard ------------------------------------------------
        # CTC loss requires: input_length >= target_length for every sample.
        # The encoder downsamples time by `subsampling_factor`, so the logit
        # sequence length Ã¢â€°Ë† n_frames // subsampling_factor.
        # If any sample violates this, skip it to avoid a CUDA illegal-memory-access.
        n_frames = inputs["attention_mask"].sum(dim=-1)  # (B,)
        logit_lengths = n_frames // self.subsampling_factor
        target_lengths = (label_ids != self.pad_token_id).sum(dim=-1)  # (B,)

        keep = logit_lengths >= target_lengths
        if not keep.all():
            keep_idx = keep.nonzero(as_tuple=True)[0]
            if len(keep_idx) == 0:
                raise ValueError(
                    "Every sample in this batch violates the CTC length "
                    "constraint (label longer than downsampled input). "
                    "Consider filtering short audio from the dataset."
                )
            inputs["input_features"] = inputs["input_features"][keep_idx]
            inputs["attention_mask"] = inputs["attention_mask"][keep_idx]
            label_ids = label_ids[keep_idx]
        # ----------------------------------------------------------------------

        # Return CPU tensors; Accelerate handles device placement
        return {
            "input_features": inputs["input_features"].to(dtype=self.dtype),
            "attention_mask": inputs["attention_mask"],
            "labels": label_ids,
        }

In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias=LORA_BIAS,
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

In [ ]:
set_seed(SEED)

# ── Data collator & loaders ──────────────────────────────────────────────
data_collator = CTCDataCollator(
    processor=processor,
    audio_column=AUDIO_COLUMN,
    text_column=TEXT_COLUMN,
    sampling_rate=SAMPLE_RATE,
    dtype=DTYPE,
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    sampler=SequentialSampler(train_dataset),
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0,
)

eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    sampler=SequentialSampler(eval_dataset),
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0,
)

# ── Accelerator ──────────────────────────────────────────────────────────
accelerator = Accelerator(
    mixed_precision="bf16",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
)

# ── Optimizer & scheduler ────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(ADAM_BETA1, ADAM_BETA2),
    eps=ADAM_EPSILON,
)

num_update_steps_per_epoch = math.ceil(len(train_dataloader) / GRADIENT_ACCUMULATION_STEPS)
max_train_steps = NUM_TRAIN_EPOCHS * num_update_steps_per_epoch

lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=max_train_steps,
)

# ── Prepare with Accelerate ─────────────────────────────────────────────
model, optimizer, train_dataloader, eval_dataloader, lr_scheduler = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader, lr_scheduler
)

# ── Helper functions ─────────────────────────────────────────────────────
EVAL_PREDICTIONS_PATH = OUTPUT_DIR / "eval_predictions.jsonl"

def _json_safe_value(value):
    if isinstance(value, np.generic):
        return value.item()
    return value

def _build_eval_export_rows(eval_ds, pred_texts, eval_step):
    references = []
    rows = []
    for idx in range(len(eval_ds)):
        example = eval_ds[idx]
        row = {
            key: _json_safe_value(value)
            for key, value in example.items()
            if key != AUDIO_COLUMN
        }
        reference_text = str(example.get(TEXT_COLUMN, ""))
        predicted_text = str(pred_texts[idx])
        row["eval_step"] = int(eval_step)
        row["reference_text"] = reference_text
        row["predicted_text"] = predicted_text
        row["example_wer"] = float(score_wer(actual=[reference_text], predicted=[predicted_text]))
        references.append(reference_text)
        rows.append(row)
    corpus_wer = float(score_wer(actual=references, predicted=pred_texts))
    return rows, corpus_wer

def _append_jsonl(path, rows):
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

@torch.no_grad()
def evaluate(model, eval_dataloader, eval_dataset, global_step):
    """Run evaluation, compute WER, and export per-example predictions."""
    model.eval()
    all_pred_ids = []
    all_label_ids = []

    for batch in tqdm(eval_dataloader, desc="Evaluating", leave=False):
        outputs = model(
            input_features=batch["input_features"],
            attention_mask=batch["attention_mask"],
        )
        logits = outputs.logits
        pred_ids = torch.argmax(logits, dim=-1)
        all_pred_ids.append(accelerator.gather_for_metrics(pred_ids).cpu().numpy())
        all_label_ids.append(accelerator.gather_for_metrics(batch["labels"]).cpu().numpy())

    all_pred_ids = np.concatenate(all_pred_ids, axis=0)
    all_label_ids = np.concatenate(all_label_ids, axis=0)

    # Decode predictions and labels
    pred_texts = processor.batch_decode(all_pred_ids)
    label_ids_clean = all_label_ids.copy()
    label_ids_clean[label_ids_clean == -100] = processor.tokenizer.pad_token_id
    label_texts = processor.tokenizer.batch_decode(label_ids_clean, group_tokens=False)

    corpus_wer = float(score_wer(actual=label_texts, predicted=pred_texts))

    # Export per-example predictions
    if accelerator.is_main_process:
        rows, export_wer = _build_eval_export_rows(eval_dataset, pred_texts, global_step)
        _append_jsonl(EVAL_PREDICTIONS_PATH, rows)

    model.train()
    return {"wer": corpus_wer}

def save_checkpoint(model, epoch, save_dir):
    """Save LoRA adapter weights, managing save_total_limit."""
    ckpt_dir = save_dir / f"checkpoint-epoch-{epoch}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    unwrapped = accelerator.unwrap_model(model)
    unwrapped.save_pretrained(str(ckpt_dir))
    processor.save_pretrained(str(ckpt_dir))

    # Enforce save_total_limit
    existing = sorted(save_dir.glob("checkpoint-epoch-*"), key=lambda p: p.stat().st_mtime)
    while len(existing) > SAVE_TOTAL_LIMIT:
        import shutil
        shutil.rmtree(existing.pop(0))

print(f"Training for {NUM_TRAIN_EPOCHS} epochs, {max_train_steps} optimizer steps")
print(f"  Batches/epoch: {len(train_dataloader)}, grad accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Warmup steps: {WARMUP_STEPS}")

In [ ]:
global_step = 0
best_wer = float("inf")

for epoch in range(1, NUM_TRAIN_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    progress = tqdm(train_dataloader, desc=f"Epoch {epoch}/{NUM_TRAIN_EPOCHS}", leave=True)

    for step, batch in enumerate(progress, start=1):
        # Original notebook ran forward pass in eval mode (BN / dropout frozen)
        model.eval()

        with accelerator.accumulate(model):
            outputs = model(
                input_features=batch["input_features"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        if accelerator.sync_gradients:
            global_step += 1

        epoch_loss += loss.detach().float().item()
        num_batches += 1

        if global_step % LOGGING_STEPS == 0:
            avg_loss = epoch_loss / num_batches
            lr = lr_scheduler.get_last_lr()[0]
            progress.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{lr:.2e}", step=global_step)

    # ── End of epoch: evaluate ────────────────────────────────────────────
    metrics = evaluate(model, eval_dataloader, eval_dataset, global_step)
    avg_epoch_loss = epoch_loss / max(num_batches, 1)
    accelerator.print(
        f"Epoch {epoch} | train_loss={avg_epoch_loss:.4f} | eval_wer={metrics['wer']:.4f}"
    )

    # ── Save checkpoint ───────────────────────────────────────────────────
    if accelerator.is_main_process:
        save_checkpoint(model, epoch, OUTPUT_DIR)

    if metrics["wer"] < best_wer:
        best_wer = metrics["wer"]

accelerator.print(f"\nTraining complete. Best WER: {best_wer:.4f}")